# Credit Risk Predictor Demo

This notebook is a preloaded grader demo for the final credit-risk model. Run all cells to load the bundled 2018 scoring sample, generate downgrade probabilities with the best Random Forest model, and review the main risk patterns without uploading any files.

## What This Notebook Does

When you run all cells, the notebook will automatically:

1. Load the repo-local processed feature dataset and best trained model.
2. Score a fixed preloaded sample from year 2018.
3. Show the highest-risk companies in the sample.
4. Summarize predicted risk by company attributes such as country and sector.
5. Display the most important Random Forest features.

No manual file selection is required.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from prediction_utils import (
    FEATURES_PATH,
    MODEL_PATH,
    BASE_DIR,
    get_model_feature_names,
    load_demo_input,
    load_reference_model,
    score_feature_dataframe,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

print('Project directory:', BASE_DIR)
print('Feature file:', FEATURES_PATH)
print('Model file:', MODEL_PATH)

In [ ]:
demo_df = load_demo_input(250)
print('Preloaded sample shape:', demo_df.shape)
print('Sample years:', sorted(demo_df['year'].unique().tolist()))
demo_df.head()

In [ ]:
scored_df, summary = score_feature_dataframe(demo_df)
summary_df = pd.DataFrame([summary])
summary_df

In [ ]:
prediction_columns = [
    col
    for col in [
        'Company name',
        'Country',
        'Region',
        'year',
        'downgrade_probability',
        'predicted_downgrade_0_5',
    ]
    if col in scored_df.columns
]

top_risk = scored_df[prediction_columns].sort_values('downgrade_probability', ascending=False).head(20)
top_risk

In [ ]:
risk_summary = pd.DataFrame({
    'rows_scored': [len(scored_df)],
    'mean_probability': [scored_df['downgrade_probability'].mean()],
    'median_probability': [scored_df['downgrade_probability'].median()],
    'high_risk_count_0_5': [(scored_df['downgrade_probability'] >= 0.5).sum()],
    'high_risk_share_0_5': [(scored_df['downgrade_probability'] >= 0.5).mean()],
})
risk_summary

In [ ]:
country_summary = (
    scored_df.groupby('Country', dropna=False)['downgrade_probability']
    .agg(['count', 'mean', 'median'])
    .sort_values('mean', ascending=False)
    .head(10)
)
country_summary

In [ ]:
sector_summary = (
    scored_df.groupby('Sector 1', dropna=False)['downgrade_probability']
    .agg(['count', 'mean'])
    .sort_values(['mean', 'count'], ascending=[False, False])
    .head(10)
)
sector_summary

In [ ]:
plt.figure(figsize=(8, 5))
scored_df['downgrade_probability'].plot(kind='hist', bins=20, color='#2f6c8f', edgecolor='white')
plt.title('Distribution of Predicted Downgrade Probability')
plt.xlabel('Predicted probability')
plt.ylabel('Number of companies')
plt.tight_layout()
plt.show()

In [ ]:
model = load_reference_model()
feature_names = get_model_feature_names()
importance = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
importance.head(10).to_frame(name='importance')

In [ ]:
plt.figure(figsize=(8, 5))
importance.head(10).sort_values().plot(kind='barh', color='#2f6c8f')
plt.title('Top 10 Random Forest Features')
plt.xlabel('Feature importance')
plt.tight_layout()
plt.show()

## Interpretation

This preloaded demo is designed to work with no manual setup beyond running the project pipeline once. The strongest risk drivers in this model are current credit quality (`mscore_num`), recent credit-score movement (`MScore_trend`), profitability (`return_on_assets`), leverage, and asset growth, which matches the broader financial intuition from the project report.